# 🛒 Olist Brazilian E-Commerce — Exploratory Data Analysis
**Author:** Shahd Ahmed Khaled | Data Science Portfolio Project

**Dataset:** [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)

**Goal:** Extract actionable business insights from 100k+ real orders (2016–2018)

---
### 🔍 Key Questions We'll Answer:
1. What is the monthly revenue trend?
2. Which product categories generate the most revenue?
3. How are orders distributed by status?
4. Which Brazilian states have the most customers?
5. How satisfied are customers? (Review score analysis)
6. How does actual delivery time compare to estimated?
7. What are the most common payment methods?
8. What is the average order value distribution?

## 1. 📦 Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Style ──────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#F8F9FA',
    'axes.grid':        True,
    'grid.color':       'white',
    'grid.linewidth':   1.2,
    'font.family':      'DejaVu Sans',
    'axes.spines.top':  False,
    'axes.spines.right':False,
})
PALETTE = ['#4361EE','#3A0CA3','#7209B7','#F72585','#4CC9F0','#4895EF','#560BAD']

print('✅ Libraries loaded successfully')

In [ ]:
# ── Load all CSV files ─────────────────────────────────
# Make sure all CSV files are in the same folder as this notebook

orders    = pd.read_csv('olist_orders_dataset.csv')
items     = pd.read_csv('olist_order_items_dataset.csv')
payments  = pd.read_csv('olist_order_payments_dataset.csv')
reviews   = pd.read_csv('olist_order_reviews_dataset.csv')
products  = pd.read_csv('olist_products_dataset.csv')
customers = pd.read_csv('olist_customers_dataset.csv')
sellers   = pd.read_csv('olist_sellers_dataset.csv')
category  = pd.read_csv('product_category_name_translation.csv')

print(f'Orders:    {orders.shape[0]:,} rows')
print(f'Items:     {items.shape[0]:,} rows')
print(f'Payments:  {payments.shape[0]:,} rows')
print(f'Reviews:   {reviews.shape[0]:,} rows')
print(f'Products:  {products.shape[0]:,} rows')
print(f'Customers: {customers.shape[0]:,} rows')

## 2. 🧹 Data Cleaning & Preprocessing

In [ ]:
# ── Check missing values across key tables ─────────────
print('=== Missing Values ===')
for name, df in [('orders', orders), ('items', items), ('payments', payments), ('reviews', reviews)]:
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if not missing.empty:
        print(f'\n{name}:')
        print(missing)
    else:
        print(f'\n{name}: ✅ No missing values')

In [ ]:
# ── Convert datetime columns ───────────────────────────
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

# ── Filter to delivered orders only (for time analysis) ─
delivered = orders[orders['order_status'] == 'delivered'].copy()

# ── Extract month-year ─────────────────────────────────
orders['order_month'] = orders['order_purchase_timestamp'].dt.to_period('M')

# ── Calculate actual delivery days ────────────────────
delivered['actual_days']    = (delivered['order_delivered_customer_date'] - delivered['order_purchase_timestamp']).dt.days
delivered['estimated_days'] = (delivered['order_estimated_delivery_date'] - delivered['order_purchase_timestamp']).dt.days
delivered['early_late']     = delivered['actual_days'] - delivered['estimated_days']

print('✅ Datetime columns converted')
print(f'✅ Delivered orders: {len(delivered):,}')

In [ ]:
# ── Build master dataframe ─────────────────────────────
# orders + items + products (with English category names) + customers

products_en = products.merge(category, on='product_category_name', how='left')

master = (
    orders
    .merge(items,     on='order_id',   how='left')
    .merge(products_en[['product_id','product_category_name_english']], on='product_id', how='left')
    .merge(customers[['customer_id','customer_state']], on='customer_id', how='left')
)

print(f'✅ Master DataFrame: {master.shape[0]:,} rows × {master.shape[1]} columns')

## 3. 📊 Exploratory Data Analysis

### Q1 — Monthly Revenue Trend

In [ ]:
monthly_revenue = (
    master
    .dropna(subset=['price'])
    .groupby('order_month')['price']
    .sum()
    .reset_index()
)
monthly_revenue['order_month'] = monthly_revenue['order_month'].astype(str)
# Remove incomplete months at edges
monthly_revenue = monthly_revenue.iloc[1:-1]

fig, ax = plt.subplots(figsize=(13, 4))
ax.fill_between(monthly_revenue['order_month'], monthly_revenue['price'],
                alpha=0.15, color=PALETTE[0])
ax.plot(monthly_revenue['order_month'], monthly_revenue['price'],
        color=PALETTE[0], linewidth=2.5, marker='o', markersize=5)

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x/1000:.0f}K'))
ax.set_xticks(range(0, len(monthly_revenue), 2))
ax.set_xticklabels(monthly_revenue['order_month'].iloc[::2], rotation=45, ha='right', fontsize=9)
ax.set_title('Monthly Revenue — Olist (2016–2018)', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('')
ax.set_ylabel('Revenue (BRL)')
plt.tight_layout()
plt.savefig('viz_1_monthly_revenue.png', dpi=150, bbox_inches='tight')
plt.show()

peak = monthly_revenue.loc[monthly_revenue['price'].idxmax()]
print(f'📌 Peak month: {peak["order_month"]}  →  R${peak["price"]:,.0f}')
print(f'📌 Total revenue: R${monthly_revenue["price"].sum():,.0f}')

### Q2 — Top 10 Product Categories by Revenue

In [ ]:
top_categories = (
    master
    .dropna(subset=['price','product_category_name_english'])
    .groupby('product_category_name_english')['price']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
top_categories.columns = ['Category', 'Revenue']
top_categories['Category'] = top_categories['Category'].str.replace('_', ' ').str.title()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(top_categories['Category'][::-1], top_categories['Revenue'][::-1],
               color=PALETTE[0], edgecolor='white', height=0.65)

for bar in bars:
    w = bar.get_width()
    ax.text(w + 8000, bar.get_y() + bar.get_height()/2,
            f'R${w/1000:.0f}K', va='center', fontsize=9)

ax.set_title('Top 10 Product Categories by Revenue', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Total Revenue (BRL)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x/1000:.0f}K'))
plt.tight_layout()
plt.savefig('viz_2_top_categories.png', dpi=150, bbox_inches='tight')
plt.show()

### Q3 — Order Status Distribution

In [ ]:
status_counts = orders['order_status'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart
wedge_props = dict(width=0.55, edgecolor='white', linewidth=2)
axes[0].pie(status_counts, labels=status_counts.index, autopct='%1.1f%%',
            colors=PALETTE, wedgeprops=wedge_props, startangle=90,
            textprops={'fontsize': 9})
axes[0].set_title('Order Status Distribution', fontweight='bold')

# Bar chart (excluding 'delivered' to see small statuses clearly)
others = status_counts[status_counts.index != 'delivered']
axes[1].bar(others.index, others.values, color=PALETTE[3], edgecolor='white')
axes[1].set_title('Non-Delivered Statuses (Detail)', fontweight='bold')
axes[1].set_xlabel('Status')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle('Order Status Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('viz_3_order_status.png', dpi=150, bbox_inches='tight')
plt.show()

delivery_rate = status_counts['delivered'] / status_counts.sum() * 100
print(f'📌 Successful delivery rate: {delivery_rate:.1f}%')

### Q4 — Customers by State (Top 10)

In [ ]:
customers_by_state = (
    customers['customer_state']
    .value_counts()
    .head(10)
    .reset_index()
)
customers_by_state.columns = ['State', 'Customers']

fig, ax = plt.subplots(figsize=(10, 4))
colors = [PALETTE[0] if i == 0 else PALETTE[4] for i in range(len(customers_by_state))]
ax.bar(customers_by_state['State'], customers_by_state['Customers'],
       color=colors, edgecolor='white')

for i, (_, row) in enumerate(customers_by_state.iterrows()):
    ax.text(i, row['Customers'] + 200, f"{row['Customers']:,}",
            ha='center', va='bottom', fontsize=8.5)

ax.set_title('Top 10 Brazilian States by Number of Customers', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('State')
ax.set_ylabel('Number of Customers')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
plt.tight_layout()
plt.savefig('viz_4_customers_by_state.png', dpi=150, bbox_inches='tight')
plt.show()

top_state = customers_by_state.iloc[0]
top_pct = top_state['Customers'] / customers_by_state['Customers'].sum() * 100
print(f'📌 Top state: {top_state["State"]} with {top_state["Customers"]:,} customers ({top_pct:.1f}% of top 10)')

### Q5 — Customer Satisfaction (Review Scores)

In [ ]:
score_counts = reviews['review_score'].value_counts().sort_index()
score_pct    = score_counts / score_counts.sum() * 100

score_colors = ['#E63946','#F4A261','#E9C46A','#90BE6D','#2D6A4F']

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(score_counts.index, score_pct, color=score_colors, edgecolor='white', width=0.65)

for bar, pct in zip(bars, score_pct):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{pct:.1f}%', ha='center', fontsize=10, fontweight='500')

ax.set_title('Customer Review Score Distribution', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Review Score (1=Worst, 5=Best)')
ax.set_ylabel('Percentage of Reviews')
ax.set_xticks([1,2,3,4,5])
ax.set_xticklabels(['1 ⭐','2 ⭐','3 ⭐','4 ⭐','5 ⭐'])
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
plt.tight_layout()
plt.savefig('viz_5_review_scores.png', dpi=150, bbox_inches='tight')
plt.show()

avg_score = reviews['review_score'].mean()
positive  = score_pct[score_pct.index >= 4].sum()
print(f'📌 Average review score: {avg_score:.2f} / 5.0')
print(f'📌 Positive reviews (4-5 stars): {positive:.1f}%')

### Q6 — Delivery Performance: Estimated vs Actual

In [ ]:
delivery_clean = delivered.dropna(subset=['actual_days','estimated_days'])
delivery_clean = delivery_clean[
    (delivery_clean['actual_days'] > 0) &
    (delivery_clean['actual_days'] < 100)
]

on_time    = (delivery_clean['early_late'] <= 0).sum()
late       = (delivery_clean['early_late'] >  0).sum()
on_time_pct = on_time / len(delivery_clean) * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Distribution comparison
axes[0].hist(delivery_clean['actual_days'],    bins=40, alpha=0.6, label='Actual days',    color=PALETTE[0])
axes[0].hist(delivery_clean['estimated_days'], bins=40, alpha=0.6, label='Estimated days', color=PALETTE[3])
axes[0].axvline(delivery_clean['actual_days'].median(),    color=PALETTE[0], linestyle='--', linewidth=1.5)
axes[0].axvline(delivery_clean['estimated_days'].median(), color=PALETTE[3], linestyle='--', linewidth=1.5)
axes[0].set_title('Actual vs Estimated Delivery Days', fontweight='bold')
axes[0].set_xlabel('Days')
axes[0].set_ylabel('Count')
axes[0].legend()

# On-time pie
axes[1].pie([on_time, late], labels=['On-Time / Early','Late'],
            colors=[PALETTE[4], PALETTE[3]], autopct='%1.1f%%',
            wedgeprops=dict(width=0.55, edgecolor='white', linewidth=2),
            startangle=90, textprops={'fontsize':10})
axes[1].set_title('Delivery On-Time Rate', fontweight='bold')

plt.suptitle('Delivery Performance Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('viz_6_delivery_performance.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'📌 On-time delivery rate: {on_time_pct:.1f}%')
print(f'📌 Median actual delivery time: {delivery_clean["actual_days"].median():.0f} days')
print(f'📌 Median estimated time:       {delivery_clean["estimated_days"].median():.0f} days')

### Q7 — Payment Methods

In [ ]:
payment_counts  = payments['payment_type'].value_counts()
payment_revenue = payments.groupby('payment_type')['payment_value'].sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# By count
axes[0].pie(payment_counts, labels=payment_counts.index, autopct='%1.1f%%',
            colors=PALETTE, wedgeprops=dict(width=0.55, edgecolor='white', linewidth=2),
            startangle=90, textprops={'fontsize': 9})
axes[0].set_title('Payment Method — by Order Count', fontweight='bold')

# By revenue
axes[1].bar(payment_revenue.index, payment_revenue.values / 1e6,
            color=PALETTE[:len(payment_revenue)], edgecolor='white')
axes[1].set_title('Payment Method — by Total Revenue', fontweight='bold')
axes[1].set_ylabel('Revenue (Million BRL)')
axes[1].set_xlabel('Payment Type')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x:.1f}M'))

plt.suptitle('Payment Method Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('viz_7_payment_methods.png', dpi=150, bbox_inches='tight')
plt.show()

top_payment = payment_counts.index[0]
top_pct     = payment_counts.iloc[0] / payment_counts.sum() * 100
print(f'📌 Most used payment method: {top_payment} ({top_pct:.1f}% of orders)')

### Q8 — Order Value Distribution

In [ ]:
order_values = (
    items.groupby('order_id')['price']
    .sum()
    .reset_index()
)
order_values.columns = ['order_id','order_total']
order_values = order_values[order_values['order_total'] < 1000]  # Remove extreme outliers

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(order_values['order_total'], bins=50, color=PALETTE[2], edgecolor='white', alpha=0.85)
axes[0].axvline(order_values['order_total'].median(), color=PALETTE[3],
                linestyle='--', linewidth=2, label=f'Median: R${order_values["order_total"].median():.0f}')
axes[0].axvline(order_values['order_total'].mean(), color=PALETTE[0],
                linestyle='--', linewidth=2, label=f'Mean: R${order_values["order_total"].mean():.0f}')
axes[0].set_title('Order Value Distribution', fontweight='bold')
axes[0].set_xlabel('Order Total (BRL)')
axes[0].set_ylabel('Number of Orders')
axes[0].legend()

# Box plot by quartile
bins   = [0, 50, 100, 200, 500, 1000]
labels = ['<50','50–100','100–200','200–500','500+']
order_values['range'] = pd.cut(order_values['order_total'], bins=bins, labels=labels)
range_counts = order_values['range'].value_counts().sort_index()
axes[1].bar(range_counts.index, range_counts.values, color=PALETTE[0], edgecolor='white')
axes[1].set_title('Orders by Value Range', fontweight='bold')
axes[1].set_xlabel('Order Total Range (BRL)')
axes[1].set_ylabel('Number of Orders')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))

plt.suptitle('Order Value Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('viz_8_order_values.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'📌 Average order value:  R${order_values["order_total"].mean():.2f}')
print(f'📌 Median order value:   R${order_values["order_total"].median():.2f}')
print(f'📌 Total orders analyzed: {len(order_values):,}')

## 4. 📝 Summary of Key Insights

In [ ]:
print('=' * 55)
print('   OLIST BRAZILIAN E-COMMERCE — KEY INSIGHTS')
print('=' * 55)

print(f"""
📈  REVENUE
    • Revenue grew steadily from late 2016 to mid-2018
    • Clear seasonality with peak in Nov (Black Friday)

🛒  PRODUCTS
    • Top category: Health & Beauty / Bed-Bath-Table
    • These two categories alone drive ~20%+ of revenue

📦  ORDERS
    • {delivery_rate:.1f}% of orders successfully delivered
    • {on_time_pct:.1f}% of deliveries arrived on time or early

📍  CUSTOMERS
    • São Paulo (SP) dominates with the most customers
    • Strong concentration in Southeast Brazil

⭐  SATISFACTION
    • Average review score: {avg_score:.2f}/5.0
    • {positive:.1f}% of customers gave 4 or 5 stars

💳  PAYMENTS
    • Credit card is overwhelmingly the #1 payment method
    • Boleto (bank slip) is the 2nd most used option
""")
print('=' * 55)

---
*Project by Shahd Ahmed Khaled — Data Science Portfolio*  
*Tools: Python, Pandas, Matplotlib, Seaborn*